## Imports

In [1]:
import os
import pandas as pd
import csv
import json
import glob

In [ ]:
# metadata = pd.read_csv('./tsv_exp_header_colhead_time_skeleton.tsv',sep='\t',nrows=10)
# metadata.head()

# df = pd.read_csv('./tsv_exp_header_colhead_skeleton.tsv',sep='\t',skiprows=10)
# pd.read_csv("mydata.csv", skiprows=10000 nrows=10000)

In [ ]:
def get_qualisys_metadata(filename:str)->dict:
    """
    Reads qualisys export file in .tsv format along with (custom) json files, parses metadata. 
    Exported file has to include tsv-header (qualisys export setting).

    Returns: metadata in dict format
    """

    qualisys_metadata = {}
    with open(filename) as fd:
        # read file with csv reader. no need for pandas for just a few lines
        reader = csv.reader(fd, delimiter="\t", quotechar='"')
        
        for ind, row in enumerate(reader):
            # only the first 9 rows of the whole file contain the tsv header metadata
            if ind <= 8:
                if ind < 6:
                    qualisys_metadata[row[0]] = float(row[1])
                # this row has 2 pieces of info; timestamp of the recording from qualisys (this cant be used for sync), timestamp from the start of host system
                elif ind == 7:
                    qualisys_metadata[row[0]+"_QUALYSIS"] = row[1]
                    qualisys_metadata[row[0]+"_FROM_SYSTEM_START"] = row[2]
                else:
                    qualisys_metadata[row[0]] = row[1]

    return qualisys_metadata

get_qualisys_metadata("./tsv_exp_header_colhead_time.tsv")

{'NO_OF_FRAMES': 6337.0,
 'NO_OF_CAMERAS': 6.0,
 'NO_OF_MARKERS': 23.0,
 'FREQUENCY': 100.0,
 'NO_OF_ANALOG': 0.0,
 'ANALOG_FREQUENCY': 0.0,
 'DESCRIPTION': '--',
 'TIME_STAMP_QUALYSIS': '2025-03-11, 14:40:58.462',
 'TIME_STAMP_FROM_SYSTEM_START': '18661.47475990',
 'DATA_INCLUDED': '3D'}

In [11]:
with open('./data_start.json','r') as f:
    content = f.read()

c = content.replace("\'", "\"")
dic = json.loads(c)
dic

{'uuid': '907A7BAA-170A-4725-8EEB-502BCB369EFA',
 'recordStatus': 'recording',
 'filename': '6AB0FAC1-DD15-41EB-A8BF-ED57403D62AA',
 'timestamp': '2025-03-11T13:38:10Z'}

In [12]:
with open('data.json', 'w') as f:
    json.dump(dic, f)

In [10]:
json_filenames = ["data_start.json", "data_stop.json"]

all_json_data = {}

for file in json_filenames:
    path = os.path.join(".",file)
    if os.path.isfile(path):
        with open(path,'r') as f:
            content = f.read()
            json_data = json.loads(content.replace("\'","\""))
            # print(json_data)
            for key,value in json_data.items():
                # print(key, value)
                if key in all_json_data.keys():
                    all_json_data[key].append(value)
                else:
                    all_json_data[key] = [value]

In [11]:
all_json_data

{'uuid': ['907A7BAA-170A-4725-8EEB-502BCB369EFA',
  '907A7BAA-170A-4725-8EEB-502BCB369EFA'],
 'recordStatus': ['recording', 'stopped'],
 'filename': ['6AB0FAC1-DD15-41EB-A8BF-ED57403D62AA',
  '6AB0FAC1-DD15-41EB-A8BF-ED57403D62AA'],
 'timestamp': ['2025-03-11T13:38:10Z', '2025-03-11T13:39:24Z'],
 'timestamp_qualisys': [1741700367242224200, 1741700367242224384]}

In [ ]:
for key,value in json_data.items():
    print(key, value)

uuid 907A7BAA-170A-4725-8EEB-502BCB369EFA
recordStatus stopped
filename 6AB0FAC1-DD15-41EB-A8BF-ED57403D62AA
timestamp 2025-03-11T13:39:24Z
timestamp_qualisys 1741700367242224384


In [9]:
json_data.keys()

dict_keys(['uuid', 'recordStatus', 'filename', 'timestamp', 'timestamp_qualisys'])

In [12]:
def read_improper_json_file(filename:str)-> dict:
    path = os.path.join(".",filename)
    if os.path.isfile(path):
        with open(path,'r') as f:
            content = f.read()
            json_data = json.loads(content.replace("\'","\""))

    return json_data

In [16]:
data_start = read_improper_json_file("data_start.json")
data_stop = read_improper_json_file("data_stop.json")

unified_json_data = {}
if data_start["uuid"] == data_stop["uuid"]:
    unified_json_data["uuid"] = data_start["uuid"]
else:
    print(f'uuid different for {data_start["filename"]}')

if data_start["filename"] == data_stop["filename"]:
    unified_json_data["filename"] = data_start["filename"]
else:
    print(f'filename different for {data_start["filename"]}')

if data_start["timestamp"] != data_stop["timestamp"]:
    unified_json_data["timestamp_start"] = data_start["timestamp"]
    unified_json_data["timestamp_stop"] = data_stop["timestamp"]
else:
    print(f'timestamps are identical for  {data_stop["timestamp"]}')

if data_start["timestamp_qualisys"] != data_stop["timestamp_qualisys"]:
    unified_json_data["timestamp_qualisys"] = data_start["timestamp_qualisys"]
    unified_json_data["timestamp_qualisys"] = data_stop["timestamp_qualisys"]
else:
    print(f'timestamp_qualisys are identical for  {data_stop["timestamp_qualisys"]}')